# Topic: Reranking & Context Compression

## Definition (30-second explanation)
*   Imagine hiring for a job with 10,000 applicants: a **Bi-Encoder** is an automated HR filter that quickly scans resumes in 1 second and shortlists the top 30 candidates based on general keywords and vibe.
*   A **Cross-Encoder (Reranker)** is the hiring manager who sits down in a room with each of those 30 candidates for a 1-on-1 deep interview, carefully comparing the exact job requirements against their specific answers to pick the single best hire.
*   **Context Compression** is the executive assistant who takes that winning candidate's 10-page portfolio and highlights only the two sentences the CEO actually needs to make the final decision.

## Why Interviewers Ask This
*   Standard vector retrieval is fast but imprecise; it frequently pulls irrelevant chunks that dilute the LLM's context window, causing the infamous "Lost in the Middle" hallucination phenomenon.
*   Interviewers want to see if you know how to build a production two-stage retrieval pipeline (retrieve broad, rank deep) to optimize retrieval recall while minimizing LLM token costs.

## Core Concepts (The 3-Layer Anatomy)
*   **The Bottleneck:** Vector search (Bi-Encoder) encodes the query and the document *independently* into static vectors, meaning the words in the query never get to interact with or attend to the words in the document. Furthermore, stuffing 10+ raw chunks into an LLM wastes expensive input tokens and increases latency.
*   **The Mechanism:**
    *   **Bi-Encoder (First Stage):** Encodes Query $Q \rightarrow \vec{u}$ and Document $D \rightarrow \vec{v}$ separately. Compares them with a cheap dot product/cosine similarity in milliseconds across millions of docs.
    *   **Cross-Encoder (Second Stage):** Concatenates $[Q, D]$ into a single string `[CLS] Query [SEP] Document` and passes it through full Transformer self-attention layers so every query token directly attends to every document token, outputting a precise relevance score between $0$ and $1$.
    *   **Context Compression / Extractive Filtering:** Passes the top reranked chunks through a lightweight LLM or compressor (like LongLLMLingua) to extract only the sentences answering the prompt, discarding conversational filler.
*   **The Trade-off:** Cross-Encoders have quadratic time complexity $\mathcal{O}((L_Q + L_D)^2)$ per chunk and cannot be pre-computed because the input depends on the dynamic query. Running a Cross-Encoder over 10,000 documents will crash latency; it should only ever touch the top 20 to 50 items.

## When to Use
*   **Two-Stage RAG Pipelines:** Standard for high-accuracy production RAG (retrieve top 30 via vector/hybrid search, rerank down to top 3-5 for the LLM).
*   **Combating "Lost in the Middle":** When an LLM fails to notice crucial facts buried in the middle of a large prompt context.
*   **Token Budget Optimization:** When passing 10 large chunks exceeds the target latency or API budget of an expensive model like GPT-4o or Claude 3.5 Sonnet.

## Advantages
*   Massive jump in Precision@K and MRR (Mean Reciprocal Rank) compared to standalone vector search.
*   Eliminates irrelevant noise, reducing LLM hallucinations and prompt token costs.
*   Allows the first-stage retriever to cast a wide, high-recall net (e.g., fetching 50 chunks) without polluting the final prompt.

## Limitations
*   Introduces extra inference latency (typically 50ms to 200ms depending on whether using Cohere API or local BGE on GPU/CPU).
*   Additional infrastructure dependency (running a local BGE reranker model or paying for Cohere Rerank API calls).
*   Cannot search an entire database directly; strictly dependent on the first-stage retriever bringing the right candidates into the candidate pool.

## Common Comparisons
*   **Bi-Encoder vs. Cross-Encoder:** Bi-Encoder embeds text separately into static vectors (fast, supports offline indexing, lower accuracy); Cross-Encoder processes query and document together via joint self-attention (slow, cannot be pre-indexed, maximum accuracy).
*   **RRF vs. Cross-Encoder Reranking:** RRF is a zero-compute mathematical heuristic sorting list ranks; a Cross-Encoder is a deep neural network reading the actual semantics of the text.

## Common Interview Traps
*   **Attempting to Index with a Cross-Encoder:** Claiming you can "store Cross-Encoder embeddings in a vector database." Cross-Encoders do NOT produce standalone embeddings; they output a single scalar score for a specific `(Query, Document)` pair.
*   **Confusing Reranking with LLM Generation:** A reranker (like `bge-reranker-large` or `cohere-rerank-v3`) is a specialized scoring encoder, not a generative autoregressive LLM.

## Python Syntax (FlashRank / Hugging Face BGE Minimal Implementation)
```python
from flashrank import Ranker, RerankRequest

# 1. Initialize lightweight cross-encoder ranker (runs ultra-fast on CPU)
ranker = Ranker(model_name="ms-marco-MiniLM-L-12-v2", cache_dir="/tmp")

# 2. Candidate passages retrieved from First-Stage Vector / Hybrid Search (Top 25)
passages = [
    {"id": 1, "text": "RTX-4090 requires a minimum 850W power supply unit."},
    {"id": 2, "text": "NVIDIA released the Ada Lovelace GPU architecture in 2022."},
]

# 3. Formulate rerank request with the user's specific query
rerank_request = RerankRequest(query="What PSU wattage do I need for 4090?", passages=passages)

# 4. Score and sort candidates via joint attention, returning only the top K
reranked_results = ranker.rerank(rerank_request)
top_chunks = reranked_results[:1]  # Pass only the highest-scoring chunk to LLM context
```

## Important Formula
*   **Bi-Encoder Similarity:**
    $$\text{Score}(Q, D) = \vec{u}_Q \cdot \vec{v}_D \quad (\text{Independent encodings, dot product})$$
*   **Cross-Encoder Scoring:**
    $$\text{Score}(Q, D) = \text{Softmax}(\mathbf{W} \cdot \text{BERT}([Q \ ; \ D])) \quad (\text{Full all-to-all cross-attention})$$

## 45-Second Interview Answer
"In production RAG, relying solely on Bi-Encoder vector search is risky because it embeds queries and documents separately, missing subtle semantic interactions and pulling in distracting noise. I architect a two-stage retrieval pipeline: first, a fast Bi-Encoder or Hybrid search casts a wide net to pull the top 30-50 candidates. Next, I pass those candidates through a Cross-Encoder reranker, such as BGE-Reranker or Cohere, which feeds the query and document together through joint self-attention layers to assign an exact relevance score. We then prune down to the top 3-5 cleanest chunks. This eliminates 'Lost in the Middle' hallucinations, slashes input token costs, and boosts retrieval precision without sacrificing system latency."